# 09. 주석과 커스터마이징

## 학습 목표
- 텍스트, 화살표로 핵심 포인트 강조
- 범례, 제목, 축 레이블 최적화
- 자동차 연비 데이터로 정책 효과 표시
- 프레젠테이션급 차트 완성

---

## 1. 주석의 중요성

**제조업 실무 사례:**
- 공정 이상 시점 표시
- 목표값·관리 한계선 표시
- 정책 변경 시점 강조
- 핵심 인사이트 시각적 강조

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.patches import Rectangle, FancyBboxPatch
import matplotlib.patches as mpatches

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터 로드
mpg = sns.load_dataset('mpg')
flights = sns.load_dataset('flights')

## 2. 기본 텍스트 주석

**비즈니스 질문:** 연비 급증 시점을 차트에 명시하려면?

In [ ]:
# 연식별 평균 연비
yearly_mpg = mpg.groupby('model_year')['mpg'].mean()
yearly_mpg.index = 1900 + yearly_mpg.index  # 전체 연도

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(yearly_mpg.index, yearly_mpg.values, 
        marker='o', linewidth=3, markersize=10, color='#3A86FF',
        markerfacecolor='#FF006E', markeredgecolor='black', markeredgewidth=1.5)

# 텍스트 주석: 오일쇼크 시점
ax.text(1973, 17, '1차 오일쇼크\n(1973)', 
        fontsize=11, fontweight='bold', color='red',
        ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.7))

# 수직선으로 시점 강조
ax.axvline(1973, color='red', linestyle='--', linewidth=2.5, alpha=0.7)

ax.set_xlabel('연식', fontsize=12)
ax.set_ylabel('평균 연비 (mpg)', fontsize=12)
ax.set_title('자동차 평균 연비 추이 및 주요 이벤트', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 텍스트 박스로 중요 시점 명확히 표시 → 인과관계 설명 용이

## 3. 화살표 주석: annotate

**비즈니스 질문:** 특정 데이터 포인트를 화살표로 가리키려면?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(yearly_mpg.index, yearly_mpg.values, 
        marker='o', linewidth=3, markersize=10, color='#3A86FF',
        markerfacecolor='#FF006E', markeredgecolor='black', markeredgewidth=1.5)

# 화살표 주석 1: 최저점
min_year = yearly_mpg.idxmin()
min_mpg = yearly_mpg.min()
ax.annotate(f'최저점\n{min_mpg:.1f} mpg', 
            xy=(min_year, min_mpg), xytext=(min_year-3, min_mpg-3),
            fontsize=11, fontweight='bold', color='blue',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightblue', alpha=0.8),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='blue'))

# 화살표 주석 2: 최고점
max_year = yearly_mpg.idxmax()
max_mpg = yearly_mpg.max()
ax.annotate(f'최고점\n{max_mpg:.1f} mpg', 
            xy=(max_year, max_mpg), xytext=(max_year-2, max_mpg+2),
            fontsize=11, fontweight='bold', color='green',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightgreen', alpha=0.8),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='green'))

# 화살표 주석 3: 급증 구간
ax.annotate('연비 급상승 구간', 
            xy=(1976, 22), xytext=(1978, 19),
            fontsize=11, fontweight='bold', color='red',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='lightyellow', alpha=0.8),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='red', 
                          connectionstyle='arc3,rad=0.3'))

ax.set_xlabel('연식', fontsize=12)
ax.set_ylabel('평균 연비 (mpg)', fontsize=12)
ax.set_title('화살표 주석을 활용한 연비 추이 분석', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 화살표로 극값과 변곡점 강조 → 청중의 시선 유도

## 4. 영역 강조: 음영 처리

**비즈니스 질문:** 특정 기간을 구간으로 표시하려면?

In [ ]:
# Flights 데이터 준비
flights['date'] = pd.to_datetime(flights['year'].astype(str) + '-' + 
                                  flights['month'].astype(str) + '-01')
flights_sorted = flights.sort_values('date')

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(flights_sorted['date'], flights_sorted['passengers'], 
        linewidth=2.5, color='#3A86FF')

# 여름 시즌 음영 (매년 6-8월)
for year in range(1949, 1961):
    summer_start = pd.to_datetime(f'{year}-06-01')
    summer_end = pd.to_datetime(f'{year}-08-31')
    ax.axvspan(summer_start, summer_end, alpha=0.2, color='orange', label='여름 시즌' if year==1949 else '')

# 1950년대 중반 강조 (급성장기)
growth_start = pd.to_datetime('1954-01-01')
growth_end = pd.to_datetime('1957-12-31')
ax.axvspan(growth_start, growth_end, alpha=0.15, color='green')
ax.text(pd.to_datetime('1956-01-01'), 480, '급성장기', 
        fontsize=13, fontweight='bold', color='darkgreen',
        ha='center', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

ax.set_xlabel('날짜', fontsize=12)
ax.set_ylabel('승객 수 (천명)', fontsize=12)
ax.set_title('항공 승객 추이 (계절 및 성장기 강조)', fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: axvspan으로 반복 패턴(계절) 및 특정 기간 시각적 구분
# 💡 여름철 승객 증가 패턴 일목요연

## 5. 수평선·수직선: 기준선 표시

**비즈니스 질문:** 목표값 및 관리 한계선을 표시하려면?

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# 제조국별 연비 분포
origins = ['usa', 'japan', 'europe']
colors = {'usa': '#E63946', 'japan': '#06FFA5', 'europe': '#457B9D'}

for origin in origins:
    data = mpg[mpg['origin'] == origin]
    ax.scatter(data['weight'], data['mpg'], 
              c=colors[origin], label=origin, alpha=0.6, s=60,
              edgecolors='black', linewidth=0.5)

# 수평선: 연비 목표 (25 mpg)
ax.axhline(25, color='red', linestyle='--', linewidth=3, label='연비 목표 (25 mpg)')

# 수평선: 평균 연비
mean_mpg = mpg['mpg'].mean()
ax.axhline(mean_mpg, color='blue', linestyle=':', linewidth=3, label=f'전체 평균 ({mean_mpg:.1f} mpg)')

# 수직선: 경량 차량 기준 (2500 lbs)
ax.axvline(2500, color='green', linestyle='-.', linewidth=3, label='경량 차량 기준 (2500 lbs)')

# 목표 달성 영역 강조
ax.fill_between([1500, 5500], 25, 50, alpha=0.1, color='green', label='목표 달성 영역')

ax.set_xlabel('무게 (lbs)', fontsize=12)
ax.set_ylabel('연비 (mpg)', fontsize=12)
ax.set_title('제조국별 연비-무게 분석 (기준선 포함)', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 일본 차량 대부분이 목표 달성 영역 + 경량 영역에 위치
# 💡 미국 차량은 목표 미달 다수 → 경량화 전략 필요

## 6. 범례 고급 커스터마이징

**개선점:** 범례 위치, 스타일, 투명도 조정

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# 실린더별 데이터
cylinders = [4, 6, 8]
cyl_colors = {4: '#06FFA5', 6: '#FFBE0B', 8: '#E63946'}
cyl_markers = {4: 'o', 6: 's', 8: '^'}

for cyl in cylinders:
    data = mpg[mpg['cylinders'] == cyl]
    ax.scatter(data['weight'], data['mpg'], 
              c=cyl_colors[cyl], marker=cyl_markers[cyl],
              s=100, alpha=0.7, edgecolors='black', linewidth=0.8,
              label=f'{cyl}기통 (n={len(data)})')

# 범례 커스터마이징
legend = ax.legend(title='실린더 수', title_fontsize=12, fontsize=11,
                  loc='upper right', frameon=True, shadow=True,
                  fancybox=True, framealpha=0.9,
                  edgecolor='black', facecolor='lightyellow')

# 범례 제목 굵게
legend.get_title().set_fontweight('bold')

ax.set_xlabel('무게 (lbs)', fontsize=12)
ax.set_ylabel('연비 (mpg)', fontsize=12)
ax.set_title('실린더별 연비-무게 분포 (고급 범례)', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 💡 인사이트: 범례에 샘플 수(n=) 표시로 데이터 신뢰성 정보 제공
# 💡 Shadow와 fancybox로 프레젠테이션 품질 향상

## 7. 제목과 축 레이블 고급 설정

**프레젠테이션급:** 다중 라인 제목, 부제목, 단위 명시

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# 연도별 평균 연비
ax.plot(yearly_mpg.index, yearly_mpg.values, 
        marker='D', linewidth=3.5, markersize=10, color='#3A86FF',
        markerfacecolor='#FF006E', markeredgecolor='black', markeredgewidth=1.5)

# 다중 라인 제목 (메인 + 서브)
ax.set_title('미국 자동차 연비 추이 분석\n1970-1982년 평균 연비 변화 및 정책 효과', 
            fontsize=15, fontweight='bold', pad=20)

# 축 레이블 (단위 명시)
ax.set_xlabel('연식 (년도)', fontsize=12, fontweight='bold', labelpad=10)
ax.set_ylabel('평균 연비 (miles per gallon)', fontsize=12, fontweight='bold', labelpad=10)

# 출처 및 주석
fig.text(0.99, 0.01, '출처: seaborn mpg 데이터셋 | 분석일: 2024', 
        ha='right', fontsize=9, style='italic', color='gray')

# 그리드 스타일
ax.grid(True, alpha=0.4, linestyle='--', linewidth=1)

# 배경색
ax.set_facecolor('#F8F9FA')

plt.tight_layout()
plt.show()

# 💡 인사이트: 제목에 맥락(기간, 목적) 포함 → 차트만으로 이해 가능
# 💡 출처 표시로 신뢰성 확보

## 8. 종합 실전: 완성형 분석 차트

**모든 기법 통합**

In [ ]:
fig, ax = plt.subplots(figsize=(14, 9))

# 데이터 플롯
ax.plot(yearly_mpg.index, yearly_mpg.values, 
        marker='o', linewidth=4, markersize=12, color='#2E86AB',
        markerfacecolor='#A23B72', markeredgecolor='white', markeredgewidth=2,
        label='연간 평균 연비', zorder=3)

# 12개월 이동평균 (부드러운 추세)
mpg_by_year_month = mpg.groupby(['model_year', 'origin'])['mpg'].mean().reset_index()
rolling_avg = yearly_mpg.rolling(window=3, center=True).mean()
ax.plot(rolling_avg.index, rolling_avg.values,
        linewidth=2.5, color='#F18F01', linestyle='--', 
        label='3년 이동평균', alpha=0.8, zorder=2)

# 1. 오일쇼크 시점 표시
ax.axvline(1973, color='red', linestyle='-.', linewidth=2.5, alpha=0.6, zorder=1)
ax.annotate('1차 오일쇼크 (1973)\n→ 연비 규제 강화', 
            xy=(1973, 17), xytext=(1975, 14),
            fontsize=11, fontweight='bold', color='darkred',
            bbox=dict(boxstyle='round,pad=0.7', facecolor='mistyrose', edgecolor='red', linewidth=2),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='red'))

# 2. 목표선
ax.axhline(25, color='green', linestyle='--', linewidth=2.5, alpha=0.7, label='CAFE 기준 (25 mpg)')

# 3. 개선 구간 음영
ax.axvspan(1973, 1982, alpha=0.12, color='green', label='집중 개선 기간')

# 4. 최저·최고점 강조
min_year, min_mpg = yearly_mpg.idxmin(), yearly_mpg.min()
max_year, max_mpg = yearly_mpg.idxmax(), yearly_mpg.max()

ax.scatter([min_year], [min_mpg], s=300, c='red', marker='v', 
          edgecolors='black', linewidth=2, zorder=4, label=f'최저: {min_mpg:.1f} mpg')
ax.scatter([max_year], [max_mpg], s=300, c='lime', marker='^', 
          edgecolors='black', linewidth=2, zorder=4, label=f'최고: {max_mpg:.1f} mpg')

# 5. 개선률 표시
improvement = max_mpg - min_mpg
improvement_pct = (improvement / min_mpg) * 100
ax.text(1976, 28, f'12년간 개선\n+{improvement:.1f} mpg ({improvement_pct:.0f}%↑)', 
        fontsize=13, fontweight='bold', color='darkgreen',
        ha='center', bbox=dict(boxstyle='round,pad=0.8', facecolor='lightgreen', 
                              edgecolor='darkgreen', linewidth=2))

# 축 및 제목 설정
ax.set_xlabel('연식', fontsize=13, fontweight='bold', labelpad=12)
ax.set_ylabel('평균 연비 (miles per gallon)', fontsize=13, fontweight='bold', labelpad=12)
ax.set_title('미국 자동차 연비 개선 추세 분석 (1970-1982)\n오일쇼크와 정책 변화가 연비에 미친 영향', 
            fontsize=16, fontweight='bold', pad=25)

# 범례
legend = ax.legend(loc='upper left', fontsize=10, frameon=True, shadow=True,
                  fancybox=True, framealpha=0.95, edgecolor='black')

# 그리드 및 스타일
ax.grid(True, alpha=0.4, linestyle=':', linewidth=1.2)
ax.set_facecolor('#FAFAFA')
ax.set_ylim(15, 30)

# 출처
fig.text(0.99, 0.01, 'Data: UCI Machine Learning Repository | Analysis: 2024', 
        ha='right', fontsize=9, style='italic', color='dimgray')

plt.tight_layout()
plt.show()

# 💡 인사이트: 모든 주석 기법 통합 → 스토리텔링 완성
# 💡 1973년 오일쇼크 → 1982년까지 연비 42% 향상
# 💡 정책·사건·데이터를 하나의 내러티브로 연결

---

## 📊 핵심 요약

### 코드 패턴 (30%)
```python
# 텍스트 주석
ax.text(x, y, 'text', fontsize=11, bbox=dict(boxstyle='round', facecolor='yellow'))

# 화살표 주석
ax.annotate('text', xy=(x, y), xytext=(x_offset, y_offset),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))

# 수평·수직선
ax.axhline(value, color='red', linestyle='--', linewidth=2)
ax.axvline(value, color='blue', linestyle='-.', linewidth=2)

# 영역 음영
ax.axvspan(xmin, xmax, alpha=0.2, color='green')
ax.fill_between(x, y1, y2, alpha=0.3, color='blue')

# 범례 커스터마이징
ax.legend(loc='upper right', frameon=True, shadow=True, fancybox=True)
```

### 주석 활용 전략 (70%)
| 목적 | 도구 | 적용 사례 |
|------|------|----------|
| 특정 값 강조 | text | 극값, 평균 표시 |
| 인과 설명 | annotate | 이벤트-결과 연결 |
| 기준 표시 | axhline/axvline | 목표, 한계선 |
| 구간 강조 | axvspan | 계절, 정책 기간 |

### 도출된 인사이트
1. **오일쇼크 효과**: 1973년 이후 9년간 연비 42% 향상 → 규제의 실효성
2. **정책 시점**: 주석으로 외부 이벤트와 데이터 변화 직접 연결
3. **계절성 패턴**: 음영 처리로 여름 피크 시즌 반복 패턴 명확화
4. **목표 달성**: 기준선으로 일본 차량의 우수성 시각적 증명

### 프레젠테이션 체크리스트
- [ ] 제목에 기간·목적·핵심 메시지 포함
- [ ] 축 레이블에 단위 명시
- [ ] 핵심 포인트에 화살표·텍스트 주석
- [ ] 기준선(목표, 평균)  표시
- [ ] 범례 위치·스타일 최적화
- [ ] 출처 및 분석 일자 표기
- [ ] 색상 일관성 (색맹 고려)

### 주의사항
- 주석 너무 많으면 가독성 저하 → 핵심 3-5개만
- 화살표 방향은 자연스럽게 (좌→우, 위→아래)
- 박스 배경색은 차트와 대비되게
- 텍스트 크기는 본문보다 작게